# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [3]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [4]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [5]:
stop_pt = {unidecode.unidecode(k) for k in stopwords.words("portuguese")}
stop_pt -= {"nao"}   # olumsuzluk korunmalı: "nao recomendo" ≠ "recomendo"

def cleaning(sentence):
    sentence = sentence.strip().lower()
    sentence = unidecode.unidecode(sentence)                     # não → nao
    sentence = "".join(k for k in sentence if not k.isdigit())
    sentence = sentence.translate(str.maketrans(string.punctuation, " " * len(string.punctuation)))
    kelimeler = word_tokenize(sentence)
    return " ".join(k for k in kelimeler if k not in stop_pt)

cleaning("Não recomendo! O produto chegou QUEBRADO, péssimo... 2 dias de atraso.")

'nao recomendo produto chegou quebrado pessimo dias atraso'

In [6]:
df["full_review_cleaned"] = df["full_review"].apply(cleaning)
df[["full_review", "full_review_cleaned"]].head()

,full_review,full_review_cleaned
0,Recebi bem antes do prazo estipulado.,recebi bem antes prazo estipulado
1,Parabéns lojas lannister adorei comprar pela ...,parabens lojas lannister adorei comprar intern...
2,recomendo aparelho eficiente. no site a marca ...,recomendo aparelho eficiente site marca aparel...
3,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",pouco travando valor ta boa
4,"Super recomendo Vendedor confiável, produto ok...",super recomendo vendedor confiavel produto ok ...


## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [7]:
oran_kotu = (df["review_score"] <= 3).mean()
print(f"1–3 puanlı yorumların oranı: %{100 * oran_kotu:.1f}")
df["review_score"].value_counts(normalize=True).sort_index()

1–3 puanlı yorumların oranı: %26.7


review_score
1    0.134447
2    0.045922
3    0.086810
4    0.161254
5    0.571567
Name: proportion, dtype: float64

🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [8]:
df = df[df["review_score"] <= 3].reset_index(drop=True)
df.shape

(9658, 7)

### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [9]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.75, max_features=5000)
X_vec = vectorizer.fit_transform(df["full_review_cleaned"])

print("Şekil:", X_vec.shape)
print("Örnek bigram'lar:", [k for k in vectorizer.get_feature_names_out() if " " in k][:15])

Şekil: (9658, 5000)
Örnek bigram'lar: ['abri caixa', 'abri embalagem', 'abri reclamacao', 'abrir embalagem', 'abrir reclamacao', 'absurdo entregue', 'acabamento deixa', 'acabamento nao', 'acabamento ruim', 'acabei comprando', 'acabei recebendo', 'achando fornecedor', 'achei bem', 'achei corda', 'achei demorou']


### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [10]:
lda = LatentDirichletAllocation(n_components=3, random_state=0)
lda.fit(X_vec)

LatentDirichletAllocation(n_components=3, random_state=0)

#### Belge Karışımı (Konular için)

In [11]:
document_mixture = lda.transform(X_vec)
document_mixture_df = pd.DataFrame(document_mixture, columns=["Topic 0", "Topic 1", "Topic 2"]).round(3)
document_mixture_df.head()

,Topic 0,Topic 1,Topic 2
0,0.085,0.348,0.568
1,0.080,0.081,0.839
2,0.370,0.557,0.073
3,0.521,0.376,0.103
4,0.744,0.128,0.129


👉 Her inceleme için en önemli konuyu rapor edelim

In [12]:
df["most_important_topic"] = document_mixture.argmax(axis=1)
df["most_important_topic"].value_counts()

most_important_topic
2    3684
1    3132
0    2842
Name: count, dtype: int64

#### Konu Karışımı (Kelimeler için)

In [13]:
topic_mixture = pd.DataFrame(lda.components_, columns=vectorizer.get_feature_names_out())
topic_mixture.shape

(3, 5000)

#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [14]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [15]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [16]:
print_topics(vectorizer, lda, 15)

--------------------
Topic 0:
[('bom', 230.75), ('prazo', 128.88), ('produto', 126.52), ('nao', 113.07), ('entrega', 90.25), ('recomendo', 84.39), ('gostei', 72.77), ('chegou', 69.63), ('veio', 62.78), ('entregue', 58.87), ('boa', 55.59), ('frete', 53.43), ('demorou', 52.45), ('antes', 49.96), ('dentro', 48.69)]
--------------------
Topic 1:
[('nao', 212.15), ('produto', 201.96), ('qualidade', 113.74), ('veio', 105.36), ('defeito', 86.22), ('produto nao', 67.46), ('ruim', 58.8), ('pessima', 50.35), ('produto veio', 49.33), ('foto', 45.22), ('funciona', 43.62), ('original', 43.28), ('veio defeito', 42.82), ('material', 41.24), ('achei', 40.57)]
--------------------
Topic 2:
[('recebi', 299.57), ('comprei', 188.71), ('nao', 176.13), ('produto', 167.59), ('nao recebi', 166.46), ('recebi produto', 119.96), ('entregue', 113.59), ('apenas', 109.08), ('pedido', 101.29), ('veio', 92.48), ('dois', 91.72), ('produtos', 87.67), ('ainda', 83.41), ('errado', 76.62), ('outro', 76.32)]


🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [17]:
topic_word_mixture = [topic_word(vectorizer, lda, topic, 10, with_weights=False)
                      for topic in lda.components_]
topic_word_mixture

[['bom',
  'prazo',
  'produto',
  'nao',
  'entrega',
  'recomendo',
  'gostei',
  'chegou',
  'veio',
  'entregue'],
 ['nao',
  'produto',
  'qualidade',
  'veio',
  'defeito',
  'produto nao',
  'ruim',
  'pessima',
  'produto veio',
  'foto'],
 ['recebi',
  'comprei',
  'nao',
  'produto',
  'nao recebi',
  'recebi produto',
  'entregue',
  'apenas',
  'pedido',
  'veio']]

In [18]:
df["most_important_words"] = df["most_important_topic"].apply(lambda i: topic_word_mixture[i])

In [19]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
0,e233e51d11511bf30e568c76360ace52,1,eletronicos,recebi somente controle midea split estilo fal...,2,"[recebi, comprei, nao, produto, nao recebi, re..."
1,8b230a1373c6dc4bd867099fda1d7039,3,ferramentas_jardim,comprei duas unidades recebi agora faco,2,"[recebi, comprei, nao, produto, nao recebi, re..."
2,cb2fc3e5711b5ae85e0491ee18af63ed,3,beleza_saude,produto bom porem veio mim nao condiz foto anu...,1,"[nao, produto, qualidade, veio, defeito, produ..."
3,60c714ed14cef913944a3147094a4742,1,moveis_decoracao,produto inferior mal acabado,0,"[bom, prazo, produto, nao, entrega, recomendo,..."
4,0bd4dcc4f6c4621baf37f73495cad8c4,3,esporte_lazer,entrega prazo,0,"[bom, prazo, produto, nao, entrega, recomendo,..."


## (3.4) Pipeline Tf-Idf ve LDA

In [20]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [21]:
pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), max_df=0.75, max_features=5000),
    LatentDirichletAllocation(n_components=3, random_state=0)
)
pipeline.fit(df["full_review_cleaned"])

Pipeline(steps=[('tfidfvectorizer',
                 TfidfVectorizer(max_df=0.75, max_features=5000,
                                 ngram_range=(1, 2))),
                ('latentdirichletallocation',
                 LatentDirichletAllocation(n_components=3, random_state=0))])

💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [22]:
pipeline._final_estimator

LatentDirichletAllocation(n_components=3, random_state=0)

In [23]:
pipeline._final_estimator.components_

array([[1.13486697, 1.93762523, 8.055396  , ..., 0.33449017, 0.40943086,
        6.6741776 ],
       [1.81876114, 5.61954066, 0.46450898, ..., 2.85438183, 0.47220257,
        0.37996615],
       [1.37010582, 0.37047217, 0.36621913, ..., 0.34366586, 8.28207135,
        0.36274462]])

Pipeline ile **Belge Karışımı**:

In [24]:
document_mixture_pipe = pipeline.transform(df["full_review_cleaned"])
print("Önceki sonuçla aynı mı:", np.allclose(document_mixture_pipe, document_mixture))
pd.DataFrame(document_mixture_pipe, columns=["Topic 0", "Topic 1", "Topic 2"]).round(3).head()

Önceki sonuçla aynı mı: True


,Topic 0,Topic 1,Topic 2
0,0.085,0.348,0.568
1,0.080,0.081,0.839
2,0.370,0.557,0.073
3,0.521,0.376,0.103
4,0.744,0.128,0.129


Pipeline ile **Konu Karışımı**:

In [25]:
topic_mixture_pipe = pd.DataFrame(
    pipeline._final_estimator.components_,
    columns=pipeline[0].get_feature_names_out()
)
topic_mixture_pipe.shape

(3, 5000)

## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [26]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

review_score                           
                                         count      mean median       std
product_category_name                                                    
consoles_games                              92  1.956522    2.0  0.924787
fashion_bolsas_e_acessorios                155  1.948387    2.0  0.917319
malas_acessorios                            66  1.924242    2.0  0.949666
papelaria                                  155  1.922581    2.0  0.922556
casa_construcao                             66  1.909091    2.0  0.836242
utilidades_domesticas                      586  1.889078    2.0  0.888473
cool_stuff                                 298  1.885906    2.0  0.906618
relogios_presentes                         589  1.881154    2.0  0.905088
pet_shop                                   147  1.857143    2.0  0.883641
moveis_escritorio                          261  1.854406    1.0  0.920843
cama_mesa_banho                           1221  1.850942    2.0  0.890235
esporte_lazer                              626  1.846645    2.0  0.899136
casa_conforto                               57  1.842105    2.0  0.902169
beleza_saude                               716  1.840782    1.0  0.908275
ferramentas_jardim                         318  1.817610    1.0  0.904635
telefonia                                  488  1.815574    2.0  0.875636
moveis_decoracao                           743  1.802153    1.0  0.888620
brinquedos                                 298  1.788591    1.0  0.890808
bebes                                      244  1.786885    1.0  0.877046
eletronicos                                248  1.778226    1.0  0.883502
construcao_ferramentas_construcao           71  1.774648    1.0  0.881511
automotivo                                 370  1.770270    1.0  0.870275
perfumaria                                 235  1.723404    1.0  0.879543
informatica_acessorios                     797  1.708908    1.0  0.867089
eletrodomesticos                            84  1.630952    1.0  0.875087
eletroportateis                             54  1.518519    1.0  0.770708

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [27]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

review_score                           
                              count      mean median       std
product_category_name                                         
informatica_acessorios          797  1.708908    1.0  0.867089
automotivo                      370  1.770270    1.0  0.870275
perfumaria                      235  1.723404    1.0  0.879543
eletrodomesticos                 84  1.630952    1.0  0.875087
eletroportateis                  54  1.518519    1.0  0.770708

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [28]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
5,ff722b4c68783a4459a3adb9bb4e1d0d,3,informatica_acessorios,produto chegou pc nao conseguiu reconhecer por...,1,"[nao, produto, qualidade, veio, defeito, produ..."
14,5f938e5f5f2e9a75710b54feeb9ea610,1,eletrodomesticos,medio peca nao serviu,1,"[nao, produto, qualidade, veio, defeito, produ..."
19,6b341682ab39af9fa00d72d7388c903b,3,automotivo,nota comprar produto correto capa interno outro,1,"[nao, produto, qualidade, veio, defeito, produ..."
24,b736ff4204044e49e39062584d06fa74,1,eletroportateis,loja anuncia produto entrega outro,2,"[recebi, comprei, nao, produto, nao recebi, re..."
29,6f885e2fd69412264d9e74e805175d53,1,informatica_acessorios,recebi bem rapido porem veio outro produto nao...,0,"[bom, prazo, produto, nao, entrega, recomendo,..."
...,...,...,...,...,...,...
9633,e9f0bde9a98ff79305964e9033ed8cd2,1,informatica_acessorios,produto chegou faltando pessimas condicoes,0,"[bom, prazo, produto, nao, entrega, recomendo,..."
9634,44380245bf1a49875542ea0e7f216986,1,informatica_acessorios,nao recebi produto aind assim receber produto ...,2,"[recebi, comprei, nao, produto, nao recebi, re..."
9642,970ae7c9df61910a0a30c38670db7ae5,3,informatica_acessorios,probleminha pasta multiuso pra notebook pedi c...,0,"[bom, prazo, produto, nao, entrega, recomendo,..."
9644,504403288b387de1c95a193909ba5dfd,3,perfumaria,chegou bem rapidinho produto razoavel,1,"[nao, produto, qualidade, veio, defeito, produ..."


### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [29]:
worst_products_reviews["most_important_topic"].value_counts()

most_important_topic
2    586
1    549
0    405
Name: count, dtype: int64

In [30]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

[2, 1, 0]

In [31]:
[topic_word_mixture[i] for i in bad_frequency]

[['recebi',
  'comprei',
  'nao',
  'produto',
  'nao recebi',
  'recebi produto',
  'entregue',
  'apenas',
  'pedido',
  'veio'],
 ['nao',
  'produto',
  'qualidade',
  'veio',
  'defeito',
  'produto nao',
  'ruim',
  'pessima',
  'produto veio',
  'foto'],
 ['bom',
  'prazo',
  'produto',
  'nao',
  'entrega',
  'recomendo',
  'gostei',
  'chegou',
  'veio',
  'entregue']]

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [38]:
from olist.seller import Seller
from olist.product import Product
from olist.data import Olist

data = Olist().get_data()
sellers = Seller().get_training_data()
sellers.columns

Index(['seller_id', 'seller_city', 'seller_state', 'delay_to_carrier',
       'wait_time', 'date_first_sale', 'date_last_sale', 'months_on_olist',
       'n_orders', 'quantity', 'quantity_per_order', 'sales',
       'share_of_one_stars', 'share_of_five_stars', 'review_score',
       'cost_of_reviews', 'revenues', 'profits'],
      dtype='object')

👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [39]:
worst_sellers = sellers[["seller_id", "review_score", "profits"]].sort_values(
    by = "profits",
    ascending = True).head(10)
worst_sellers

,seller_id,review_score,profits
769,6560211a19b47992c3666cc44a7e94c0,3.937093,-21439.517
2358,4a3ca9315b744ce9f8e9374361493884,3.827873,-17752.708
1357,cc419e0650a3c5ba77189a1882b7556a,4.077586,-17581.158
945,ea8482cd71df3c1969d7b9473ff13abc,3.997371,-15542.248
315,8b321bb669392f5163d04c59e235e066,4.074547,-12886.431
453,1f50f920176fa81dab994f9023523100,4.130342,-9926.079
1213,d2374cbcbb3ca4ab1086534108cc3ab7,3.751880,-8937.608
1132,7c67e1448b00f6e969d365cea6b010ab,3.488798,-7937.611
2023,cca3071e3e9bb7d12640c9fbe2301306,3.846369,-7379.011
2687,1835b56ce799e6a4dc4eddc053f04066,3.637441,-7135.579


### (5.2) En kötü satıcılar tarafından satılan ürünler

In [40]:
products = Product().get_training_data() [["product_id", "category"]]
products

,product_id,category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares
...,...,...
31479,a0b7d5a992ccda646f2d34e418fff5a0,furniture_decor
31480,bf4538d88321d0fd4412a93c974510e6,construction_tools_lights
31481,9a7c6041fa9592d9d9ef6cfe62a71f8c,bed_bath_table
31482,83808703fc0706a22e264b9d75f04a2e,computers_accessories


❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [41]:
sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

,seller_id,category
0,48436dade18ac8b2bce089ec2a041202,cool_stuff
1,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop
2,5b51032eddd242adc84c38acab88f23d,furniture_decor
3,9d7a1d34a5052409006425275ba1c2b4,perfumery
4,df560393f3a51e74553ab94004ba5c87,garden_tools
...,...,...
112645,b8bc237ba3788b23da09c0f1f3a3288c,housewares
112646,f3c38ab652836d21de61fb8314b69182,computers_accessories
112647,c3cfdc648177fdbbbb35635a37472c53,sports_leisure
112648,2b3e4a2a3ea8e01938cabda2a3e5cc79,computers_accessories


In [42]:
sellers_product_category.groupby("seller_id").count()

,category
seller_id,
0015a82c2db000af6aaaf3ae2ecb0532,3
001cca7ae9ae17fb1caed9dfb1094831,239
001e6ad469a905060d959994f1b41e4f,0
002100f778ceb8431b7a1020ff7ab48f,55
003554e2dce176b5555353e4f3555ac8,0
...,...
ffcfefa19b08742c5d315f2791395ee5,0
ffdd9f82b9a447f6f8d4b91554cc7dd3,20
ffeee66ac5d5a62fe688b9d26f83f534,14


### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [43]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [44]:
bad_reviews_sellers = worst_products_reviews.merge(data["order_items"])
bad_reviews_sellers.head(3)

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned,most_important_topic,most_important_words,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ff722b4c68783a4459a3adb9bb4e1d0d,66,3,b8702f50a689621ab21ef200b55a5b8d,informatica_acessorios,"Produto chegou, mas meu PC não conseguiu reco...",produto chegou pc nao conseguiu reconhecer por...,1,"[nao, produto, qualidade, veio, defeito, produ...",1,e76438822c17316d276c27e52e588931,5656537e588803a555b8eb41f07a944b,2018-01-10 22:07:28,24.89,17.63
1,5f938e5f5f2e9a75710b54feeb9ea610,17,1,d9ff0185a300435402f043124f530274,eletrodomesticos,Médio A peça não serviu,medio peca nao serviu,1,"[nao, produto, qualidade, veio, defeito, produ...",1,082175ee534338e4fd967076a6f848e5,d91fb3b7d041e83b64a00a3edfb37e4f,2018-05-10 16:54:16,52.00,18.24
2,6b341682ab39af9fa00d72d7388c903b,66,3,fd65651f855608ad1aa52ff4a741c31a,automotivo,Nota3 Comprar um produto correto na capa mas i...,nota comprar produto correto capa interno outro,1,"[nao, produto, qualidade, veio, defeito, produ...",1,1dce3fd38c13a3e74fe5000f33858442,72c5da29406b4234927b81855e7b64f6,2018-05-07 05:30:57,229.99,19.49


In [45]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [46]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


--------------------------------------------------
Focusing on the seller #6560211a19b47992c3666cc44a7e94c0...
seller_id                         category                 
6560211a19b47992c3666cc44a7e94c0  watches_gifts                1624
                                  fashion_bags_accessories      340
                                  audio                          32
                                  perfumery                      13
                                  computers_accessories          12
                                  sports_leisure                  7
                                  construction_tools_safety       1
Name: count, dtype: int64
['nao', 'produto', 'qualidade', 'veio', 'defeito', 'produto nao', 'ruim', 'pessima', 'produto veio', 'foto']
--------------------------------------------------
Focusing on the seller #4a3ca9315b744ce9f8e9374361493884...
seller_id                         category       
4a3ca9315b744ce9f8e9374361493884  bed_bath_table     1566

🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.